# EasyRAG Answer Grounding And Faithfulness

## Chapter position

This chapter closes the loop. It separates retrieval quality from answer quality and shows how to turn runtime behavior into measurements you can actually debug.

## Learning question

How do citation presence, support ratio, and abstention checks describe grounded answer quality?

## Success criteria

- run `evaluate_answers()` on answerable and abstain cases
- inspect citation presence, support ratio, and abstain behavior
- compare strict and loose answer settings on the same question set

## Source code anchors

- `easyrag.rag.evaluation.runner.evaluate_answers`
- `easyrag.rag.evaluation.grounding.answer_has_citations`
- `easyrag.rag.evaluation.grounding.sentence_support_ratio`
- `easyrag.rag.evaluation.grounding.answer_abstained`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.types.EvalCase`


## Method principles

- `easyrag.rag.evaluation.runner.evaluate_answers`: This runner executes answering for each eval case and checks the returned text for grounding-related signals instead of relying on pure fluency.
- `easyrag.rag.evaluation.grounding.answer_has_citations`: This check looks for citation markers in the answer text. It is a lightweight way to ask whether the answer stayed visibly tied to evidence.
- `easyrag.rag.evaluation.grounding.sentence_support_ratio`: This heuristic estimates how much of the answer is supported by the provided evidence snippets. It works sentence by sentence instead of collapsing everything into one score.
- `easyrag.rag.evaluation.grounding.answer_abstained`: This heuristic checks whether the answer explicitly abstained. It matters because a grounded system should sometimes refuse rather than overclaim.
- `easyrag.rag.types.AnswerParam`: This dataclass is the answer-side policy bundle. It controls citation budget, context budget, answer style, and abstention rules without changing retrieval behavior.
- `easyrag.rag.types.EvalCase`: This dataclass defines one evaluation expectation. It ties a question to expected documents, snippets, abstention behavior, and optional reference text so evaluation stays deterministic.


## How the code fits together

The flow in this notebook is answer result -> citation checks -> support ratio -> abstention check. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation import evaluate_answers
from easyrag.rag.evaluation.grounding import (
    answer_abstained,
    answer_has_citations,
    sentence_support_ratio,
)

## What this cell is proving

We will evaluate one answerable question and one abstain case first. After that, we will change the answer settings and inspect how the grounding checks respond.

In [ ]:
grounding_tmp = tempfile.TemporaryDirectory()
grounding_rag = EasyRAG(
    working_dir=grounding_tmp.name,
    workspace="grounding-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(grounding_rag.initialize_storages())
run_sync(
    grounding_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n",
        ],
        ids=["doc::retrieval", "doc::storage"],
        file_paths=["docs/retrieval.md", "docs/storage.md"],
    )
)

grounding_cases = [
    EvalCase(question="How does EasyRAG use query rewrite?", expected_to_abstain=False),
    EvalCase(question="When is the company retreat?", expected_to_abstain=True),
]
strict_param = AnswerParam(
    require_citations=True, allow_abstain=True, max_citations=3, max_context_chars=220
)
grounding_report = run_sync(
    evaluate_answers(
        grounding_rag,
        grounding_cases,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
        strict_param,
    )
)

print("Aggregate grounding metrics")
_print_json(grounding_report["metrics"])
print("\nPer-case grounding reports")
_print_json(grounding_report["cases"])

## Why this output looks like this

The next cell compares two answer settings on the same answerable question: one strict configuration that requires citations and allows abstention, and one looser configuration that disables citation markers and uses a tighter evidence budget.

In [ ]:
answerable_question = "How does EasyRAG use query rewrite?"
strict_answer = run_sync(
    grounding_rag.aanswer(
        answerable_question,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
        strict_param,
    )
)
loose_param = AnswerParam(
    require_citations=False, allow_abstain=False, max_citations=1, max_context_chars=90
)
loose_answer = run_sync(
    grounding_rag.aanswer(
        answerable_question,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
        loose_param,
    )
)

strict_checks = {
    "answer": strict_answer.answer,
    "has_citations": answer_has_citations(strict_answer.answer),
    "support_ratio": sentence_support_ratio(
        strict_answer.answer,
        [citation.get("snippet", "") for citation in strict_answer.selected_citations],
    ),
    "abstained": answer_abstained(strict_answer.answer),
}
loose_checks = {
    "answer": loose_answer.answer,
    "has_citations": answer_has_citations(loose_answer.answer),
    "support_ratio": sentence_support_ratio(
        loose_answer.answer,
        [citation.get("snippet", "") for citation in loose_answer.selected_citations],
    ),
    "abstained": answer_abstained(loose_answer.answer),
}
_print_json({"strict": strict_checks, "loose": loose_checks})

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

When provider-backed retrieval is available, the same answer evaluation helpers still work. The answer synthesis step may still use the built-in grounded fallback unless you explicitly pass an `answer_model_func`.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-grounding-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval supports answer faithfulness.\n",
                    "# Policy\nCitation-aware answers stay traceable.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_report = run_sync(
            evaluate_answers(
                provider_rag,
                [
                    EvalCase(
                        question="How do answers stay traceable?",
                        expected_to_abstain=False,
                    )
                ],
                QueryParam(mode="hybrid", chunk_top_k=2),
                AnswerParam(),
            )
        )
        _print_json(provider_report["metrics"])
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- A single metric rarely tells you which stage broke. Keep retrieval and answer quality separate.
- Eval cases should be small enough to audit by hand. If the expectation is vague, the metric will be vague too.
- Use metrics to narrow the search space, then inspect the case-level reports and runtime metadata.

## Takeaway

Retrieval gives you evidence. Grounding evaluation asks whether the answer actually stays inside that evidence. Citation presence is the easiest check. Support ratio tells you how much of the answer is covered by retrieved snippets. Abstain accuracy tells you whether the system knows when it should stop guessing.

In [ ]:
run_sync(grounding_rag.finalize_storages())
grounding_tmp.cleanup()
print("Cleaned up the deterministic grounding workspace.")

## Where to go next

- Continue with [08_01_code_map_and_runtime_flow.ipynb](../08_system_architecture/08_01_code_map_and_runtime_flow.ipynb) to reconnect the learning path to the implementation modules that own these stages.
- Read [07-optimization-overview.md](../../docs/07-optimization-overview.md) to see why EasyRAG treats evaluation as the prerequisite for optimization.
- Keep [06-evaluation-overview.md](../../docs/06-evaluation-overview.md) nearby as the chapter-level reference for retrieval and answer evaluation.